# TP 2 — Exercice 1 : Bruit d'image (Gaussien additif & Impulsionnel Poivre et Sel)

1. Charger `tp1_tp2_cameraman_gray.bmp` dans `im1`.
2. Appliquer un bruit blanc Gaussien de moyenne nulle et de variance $\sigma^2 = 100$ ($\sigma = 10$) $\rightarrow$ `im2`, puis étudier la variation de $\sigma^2$.
3. Implémenter `sp_noise(image, prob)` et appliquer un bruit poivre et sel avec $p = 0.10$ ($10\%$) $\rightarrow$ `im3`, puis étudier la variation de $p$.
4. Comparer visuellement et quantitativement les deux types de bruits.

In [ ]:
from pathlib import Path
import math
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "tp1_tp2_images").is_dir():
            return candidate
    raise FileNotFoundError(
        "Impossible de trouver le dossier 'tp1_tp2_images'. "
        "Lancez Jupyter depuis la racine TAI ou depuis tp1_labs / tp2_labs."
    )

ROOT_DIR = find_project_root()
IMG_DIR = ROOT_DIR / "tp1_tp2_images"
OUT_DIR = ROOT_DIR / "tp2_labs" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_image(filename, flags=cv2.IMREAD_COLOR):
    path = IMG_DIR / filename
    img = cv2.imread(str(path), flags)
    if img is None:
        raise FileNotFoundError(f"Image introuvable ou illisible: {path}")
    return img

def show_images(items, cols=3, figsize_per_cell=(5, 4)):
    n = len(items)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_cell[0], rows * figsize_per_cell[1]))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    for ax, (title, img) in zip(axes, items):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        elif img.shape[2] == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif img.shape[2] == 4:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"IMG_DIR  : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"OUT_DIR  : {OUT_DIR}")

In [ ]:
# 1) Lecture de l'image originale en niveaux de gris
im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)
print(f"Dimensions : {im1.shape}, dtype : {im1.dtype}")

def add_gaussian_noise(image, variance, seed=42):
    cv2.setRNGSeed(seed)
    sigma = math.sqrt(variance)
    noise = np.zeros(image.shape, dtype=np.float32)
    cv2.randn(noise, 0.0, sigma)
    noisy = cv2.add(image.astype(np.float32), noise)
    return np.clip(noisy, 0, 255).astype(np.uint8)

def sp_noise(image, prob, seed=42):
    rng = np.random.default_rng(seed)
    output = image.copy()
    if image.ndim == 2:
        black, white = 0, 255
    elif image.shape[2] == 3:
        black = np.array([0, 0, 0], dtype=np.uint8)
        white = np.array([255, 255, 255], dtype=np.uint8)
    else:
        black = np.array([0, 0, 0, 255], dtype=np.uint8)
        white = np.array([255, 255, 255, 255], dtype=np.uint8)
    probs = rng.random(output.shape[:2])
    output[probs < (prob / 2.0)] = black
    output[probs > 1.0 - (prob / 2.0)] = white
    return output

# 2) Bruit Gaussien variance sigma^2 = 100 (sigma = 10)
im2 = add_gaussian_noise(im1, variance=100.0, seed=42)
out_gauss = OUT_DIR / "tp2_exo1_cameraman_gauss_var100.bmp"
cv2.imwrite(str(out_gauss), im2)

# 3) Bruit Poivre et Sel p = 0.10 (10%)
im3 = sp_noise(im1, prob=0.10, seed=42)
out_sp = OUT_DIR / "tp2_exo1_cameraman_salt_pepper_p10.bmp"
cv2.imwrite(str(out_sp), im3)

print(f"Sauvegardés :\n  - {out_gauss}\n  - {out_sp}")
print(f"PSNR(im1, im2 Gauss var=100) : {cv2.PSNR(im1, im2):.2f} dB")
print(f"PSNR(im1, im3 S&P p=0.10)    : {cv2.PSNR(im1, im3):.2f} dB")

show_images([
    ("Originale (im1)", im1),
    ("Bruit Gaussien var=100 (im2)", im2),
    ("Bruit Poivre & Sel p=10% (im3)", im3),
], cols=3)

In [ ]:
# Variation de sigma^2 (bruit Gaussien) et de p (bruit Poivre & Sel)
variances = [25, 100, 400, 900]
probs = [0.02, 0.05, 0.10, 0.25]

gauss_items = [(f"Gauss var={v} (PSNR={cv2.PSNR(im1, g):.1f} dB)", g)
               for v in variances for g in [add_gaussian_noise(im1, v)]]
sp_items = [(f"S&P p={p*100:.0f}% (PSNR={cv2.PSNR(im1, s):.1f} dB)", s)
            for p in probs for s in [sp_noise(im1, p)]]

show_images(gauss_items, cols=4)
show_images(sp_items, cols=4)

### Observations et comparaison des deux types de bruits

- **Bruit blanc additif Gaussien (`im2`) :** Affecte **tous les pixels** de l'image avec une petite fluctuation aléatoire centrée en 0. À faible variance ($\sigma^2 = 25$), l'image reste très lisible ; plus $\sigma^2$ augmente ($100 \to 400 \to 900$), plus un grain uniforme apparaît dans les zones homogènes et dégrade les détails fins.
- **Bruit impulsionnel Poivre et Sel (`im3`) :** N'altère qu'une fraction $p$ des pixels en les remplaçant brutalement par des valeurs extrêmes (`0` noir ou `255` blanc), tandis que $(1-p)$ des pixels conservent exactement leur intensité d'origine. Même à $p=10\%$, le PSNR chute plus fortement ($\sim 15\text{ dB}$) que le bruit gaussien $\sigma^2=100$ ($\sim 28\text{ dB}$) en raison de l'amplitude maximale des erreurs quadratiques $(255)^2$.